In [5]:
--Adolfo David Romero
--991555778
--Assignment 3B (bonus)
USE section19
GO

Commands completed successfully.

Total execution time: 00:00:00.010

In [6]:
-- Drop all tables for testing
DROP TABLE IF EXISTS invprod;
DROP TABLE IF EXISTS invoice;
DROP TABLE IF EXISTS orderprod;
DROP TABLE IF EXISTS salesorder;
DROP TABLE IF EXISTS customer;
DROP TABLE IF EXISTS salesrep;
DROP TABLE IF EXISTS part;
GO

--using code created from assignment 2 (the i missed)
CREATE TABLE part(
  partno      CHAR(6)
, partdesc    VARCHAR(30)
, onhand      INTEGER
, partclass   CHAR(2)
, unitprice   DECIMAL(6,2)
, CONSTRAINT pkpart           PRIMARY KEY (partno)
, CONSTRAINT ckpart_partclass CHECK (partclass IN ('AP','HW','KI','SP','XX'))
);

CREATE TABLE salesrep(
  srepno      CHAR(3)
, srepname    VARCHAR(30)
, srepstreet  VARCHAR(30)
, srepcity    VARCHAR(30) CONSTRAINT nlsalesrep_srepcity NOT NULL
, srepprov    VARCHAR(3)  CONSTRAINT nlsalesrep_srepprov NOT NULL
, sreppcode   VARCHAR(6)  CONSTRAINT nlsalesrep_srepcode NOT NULL
, totcomm     DECIMAL(8,2)
, commrate    DECIMAL(3,2)
, CONSTRAINT pksalesrep PRIMARY KEY (srepno)
);

CREATE TABLE customer(
  custno      CHAR(6)
, custname    VARCHAR(30) CONSTRAINT nlcustomer_custname NOT NULL
, custstreet  VARCHAR(30) CONSTRAINT nlcustomer_custstreet NOT NULL
, custcity    VARCHAR(30) CONSTRAINT nlcustomer_custcity NOT NULL
, custprov    VARCHAR(3)  CONSTRAINT nlcustomer_custprov NOT NULL
, custpcode   VARCHAR(6)  CONSTRAINT nlcustomer_custpcode NOT NULL
, disc        DECIMAL(3,1)
, balance     DECIMAL(7,2)
, credlimit   DECIMAL(5)
, srepno      CHAR(3)
, CONSTRAINT pkcustomer        PRIMARY KEY (custno)
, CONSTRAINT fkcustomer_srepno FOREIGN KEY (srepno)    REFERENCES salesrep(srepno)
);

CREATE TABLE salesorder(
  orderno     CHAR(5)
, orderdate   DATE
, custno      CHAR(6) CONSTRAINT nlsalesorder_custno NOT NULL
, CONSTRAINT pksalesorder        PRIMARY KEY (orderno)
, CONSTRAINT fksalesorder_custno FOREIGN KEY (custno)    REFERENCES customer(custno)
);

CREATE TABLE orderprod(
  orderno     CHAR(5)
, partno      CHAR(6)
, orderqty    INTEGER
, orderprice  DECIMAL(7,2)
, CONSTRAINT pkorderprod          PRIMARY KEY (orderno, partno)
, CONSTRAINT fkorderprod_orderno  FOREIGN KEY (orderno) REFERENCES salesorder(orderno)
, CONSTRAINT fkorderprod_partno   FOREIGN KEY (partno)  REFERENCES part(partno)
, CONSTRAINT ckorderprod_orderqty CHECK (orderqty > 0 )
);

CREATE TABLE invoice
(invno       CHAR(6),
 invdate     DATE,
 orderno     Char(5) CONSTRAINT nlinvoice_orderno NOT NULL,
 CONSTRAINT pkinvoice         PRIMARY KEY (invno),
 CONSTRAINT fkinvoice_orderno FOREIGN KEY (orderno)   REFERENCES salesorder(orderno)
);

CREATE TABLE invprod
(Invno       CHAR(6),
partno       CHAR(6),
Shipqty      INTEGER,
CONSTRAINT pkinvprod  PRIMARY KEY (Invno, partno),
CONSTRAINT fk1invprod FOREIGN KEY (Invno)      REFERENCES invoice(invno),
CONSTRAINT fk2invprod FOREIGN KEY (partno)     REFERENCES part(partno)
);

--load tables in load-tables.sql

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.060

In [7]:
--verify tables
SELECT TOP 5 * FROM part 
SELECT TOP 5 * FROM salesrep 
SELECT TOP 5 * FROM customer 
SELECT TOP 5 * FROM salesorder 
SELECT TOP 5 * FROM orderprod
SELECT TOP 5 * FROM invoice
SELECT TOP 5 * FROM invprod 
GO


(5 rows affected)

(5 rows affected)

(5 rows affected)

(5 rows affected)

(5 rows affected)

...

custno,custname,custstreet,custcity,custprov,custpcode,disc,balance,credlimit,srepno
001,Violet Jones,1146 Shirley Ave.,Brampton,ON,HOHOHO,50.0,0.00,9000,SR2
002,Joan Volmer,13-1066 7th Avenue,New York,NY,906137,0.0,9000.00,0,NULL
1,Connor,Morrison Creek,Oakville,ON,L7S2J5,5.0,0.00,1000,SR5
10,Amiel,160 Fake Street,Hamilton,ON,L8N3V2,10.0,0.00,1000,SR5
101,Priya,4703 Empire Crescent,Missisauga,ON,L5R1M8,2.5,150.00,450,SR1


orderno,orderdate,custno
00123,2010-11-11,157
00250,2010-11-25,157
0055,2011-11-06,ND1
0056,2011-11-07,ND1
1,2011-12-30,1


orderno,partno,orderqty,orderprice
00123,1001,500,4.50
00250,1002,300,8.50
0055,N13,15,20000.00
0056,N14,20,30000.00
1,CH1,2,64.00


invno,invdate,orderno
00255,2010-11-09,00123
00SJM,2010-11-12,SJM00
1,2011-12-30,1
10,2011-11-13,1000
10279,2011-10-11,O2777


Invno,partno,Shipqty
00255,1002,800
00SJM,SJM0,1
1,CH1,2
10,101,2
10279,2710,5


# **PART A - CURSORS**

In [8]:
--Update the user's balance to deal with "Arithmetic overflow error"
ALTER TABLE customer
ALTER COLUMN balance DECIMAL(10,2);

DROP PROCEDURE IF EXISTS AdjustAccountBalances
GO

--Begin Cursor 
CREATE PROCEDURE AdjustAccountBalances
AS 
BEGIN
    DECLARE @custno CHAR(6), @totalAmount DECIMAL(10,2); -- Store customer number and total transaction amount (high range to deal with "Arithmetic overflow error")
    DECLARE customer_cursor CURSOR FOR SELECT custno FROM customer --customer cursor 

    OPEN customer_cursor; --executes above 'SELECT custno FROM Customer'
    FETCH NEXT FROM customer_cursor INTO @custno; --fetch the first customer

    WHILE @@FETCH_STATUS = 0 --loop through all customers (Iterate through each customer's sales transactions.)
    BEGIN

        --calculates total sales for the customer, cost per product
        SELECT @totalAmount = ISNULL(SUM(op.orderqty * op.orderprice), 0) -- ISNULL is used in case cust has NO orders (potential edge case). Resurns 0 if null
        FROM salesorder so --use alliases to acccess
        JOIN orderprod op ON so.orderno = op.orderno
        WHERE so.custno = @custno

        PRINT CONCAT('UPDATED TOTAL AMOUNT: ',@totalAmount);

        --update customer balance col
        UPDATE customer
        SET balance = balance - @totalAmount --subtract balance using var
        WHERE custno = @custno 

        PRINT CONCAT('UPDATED CUSTOMER: ',@custno);

        FETCH NEXT FROM customer_cursor INTO @custno; -- Move to next customer
    END;

    --clean up crew
    CLOSE customer_cursor;
    DEALLOCATE customer_cursor;

END;

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.063

In [9]:
--Test Cursor

--Customer 6 has an initial balance of 0 for a test case
SELECT TOP 5 * FROM customer --before
SELECT TOP 5 * FROM salesorder 
SELECT TOP 5 * FROM orderprod

EXEC AdjustAccountBalances;

SELECT TOP 5 * FROM customer --after


(5 rows affected)

(5 rows affected)

(5 rows affected)

UPDATED TOTAL AMOUNT: 10590.00

(1 row affected)

...

Total execution time: 00:00:00.364

custno,custname,custstreet,custcity,custprov,custpcode,disc,balance,credlimit,srepno
001,Violet Jones,1146 Shirley Ave.,Brampton,ON,HOHOHO,50.0,0.00,9000,SR2
002,Joan Volmer,13-1066 7th Avenue,New York,NY,906137,0.0,9000.00,0,NULL
1,Connor,Morrison Creek,Oakville,ON,L7S2J5,5.0,0.00,1000,SR5
10,Amiel,160 Fake Street,Hamilton,ON,L8N3V2,10.0,0.00,1000,SR5
101,Priya,4703 Empire Crescent,Missisauga,ON,L5R1M8,2.5,150.00,450,SR1


orderno,orderdate,custno
00123,2010-11-11,157
00250,2010-11-25,157
0055,2011-11-06,ND1
0056,2011-11-07,ND1
1,2011-12-30,1


orderno,partno,orderqty,orderprice
00123,1001,500,4.50
00250,1002,300,8.50
0055,N13,15,20000.00
0056,N14,20,30000.00
1,CH1,2,64.00


custno,custname,custstreet,custcity,custprov,custpcode,disc,balance,credlimit,srepno
001,Violet Jones,1146 Shirley Ave.,Brampton,ON,HOHOHO,50.0,-10590.00,9000,SR2
002,Joan Volmer,13-1066 7th Avenue,New York,NY,906137,0.0,8829.31,0,NULL
1,Connor,Morrison Creek,Oakville,ON,L7S2J5,5.0,-128.00,1000,SR5
10,Amiel,160 Fake Street,Hamilton,ON,L8N3V2,10.0,-22.99,1000,SR5
101,Priya,4703 Empire Crescent,Missisauga,ON,L5R1M8,2.5,-11100.00,450,SR1


# **PART B - DYNAMIC SQL**

In [10]:
DROP PROCEDURE IF EXISTS GenerateSalesAnalysisReport;
GO

CREATE PROCEDURE GenerateSalesAnalysisReport
    @Columns NVARCHAR(MAX), --user defined columns
    @Min DATE = NULL, --NULL for optional
    @Max DATE = NULL,
    @SrepName NVARCHAR(50) = NULL
AS 
BEGIN 
    DECLARE @DynamicSQL NVARCHAR(MAX);

    --Build dynamic query 
    SET @DynamicSQL = N'SELECT ' + @Columns + '
FROM SALESORDER so
JOIN ORDERPROD op ON so.orderno = op.orderno
JOIN PART p ON op.partno = p.partno
JOIN CUSTOMER c ON so.custno = c.custno
JOIN SALESREP sr ON c.srepno = sr.srepno
WHERE 1 = 1';

    --optional parameters and error catching 
    IF @Min IS NOT NULL
        SET @DynamicSQL +=' AND so.orderdate >= @Min';
    IF @Max IS NOT NULL 
        SET @DynamicSQL +=' AND so.orderdate <= @Max';
    IF @SrepName IS NOT NULL 
        SET @DynamicSQL +=' AND sr.srepname = @SrepName';

    --Execute
    EXEC sp_executesql
        @DynamicSQL,
        N'@Min DATE, @Max DATE, @SrepName NVARCHAR(50)',
        @Min = @Min,
        @Max = @Max,
        @SrepName = @SrepName;
END;



Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.022

In [11]:
--test
EXEC GenerateSalesAnalysisReport 'so.orderno, so.orderdate, c.custname, p.partdesc, op.orderqty, op.orderprice, sr.srepname'
EXEC GenerateSalesAnalysisReport 
    @Columns = 'so.orderno, so.orderdate, sr.srepname',
    @Min = '2025-01-01',
    @Max = '2025-12-31';

(333 rows affected)

(0 rows affected)

Total execution time: 00:00:00.041

orderno,orderdate,custname,partdesc,orderqty,orderprice,srepname
123CB,2011-11-14,CRAIG BONIN,HAMMER,2,5.99,William S. Burroughs
124CB,2011-11-11,CRAIG BONIN,SCREWDRIVER,2,4.99,William S. Burroughs
456OF,2011-11-14,Young kyu Han,Golf clubs,1,149.99,Pascale Desjardin
101,2000-11-04,Shubhankar,books,12,420.00,William S. Burroughs
00123,2010-11-11,Albert Manos,MC NUT METRIC,500,4.50,HARRY POTTER
00250,2010-11-25,Albert Manos,MC METRIC SOCKET,300,8.50,HARRY POTTER
1000,2011-11-12,Kevin Teixeira,Hockey Sticks,2,25.00,HARRY POTTER
1001,2011-11-12,Kevin Teixeira,Microwave,1,150.00,HARRY POTTER
36673,2009-11-10,Anas Mayat,Stove,86,499.99,Jackie Chan
6j53f,2010-11-09,Lucas Lomnicki,Sink,4,4273.72,Pascale Desjardin


orderno,orderdate,srepname
